In [4]:

# ==============================================================
# DrnDG-GWO + KNN FEATURE SELECTION
# Dataset: Breast Cancer Wisconsin
# ==============================================================

import numpy as np
import pandas as pd
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==============================================================
# 1. PARAMETERS
# ==============================================================

RANDOM_STATE = 42

N_WOLVES = 12
MAX_ITERATIONS = 40

ALPHA = 0.80
BETA = 0.20

N_FOLDS = 5
K_NEIGHBORS = 5

# Faster Random Differential Grouping
N_RNDG_SAMPLES = 2
INTERACTION_THRESHOLD = 0.01

rng = np.random.default_rng(RANDOM_STATE)


# ==============================================================
# 2. LOAD DATASET
# ==============================================================

dataset = load_breast_cancer()

X = pd.DataFrame(
    dataset.data,
    columns=dataset.feature_names
)

y = pd.Series(dataset.target)

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print("Dataset: Breast Cancer")
print("Total samples:", X.shape[0])
print("Total features:", X.shape[1])


# ==============================================================
# 3. TRAIN TEST SPLIT
# ==============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# ==============================================================
# 4. KNN CLASSIFIER
# ==============================================================

def create_knn():

    return Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(
            n_neighbors=K_NEIGHBORS
        ))
    ])


# ==============================================================
# 5. FITNESS FUNCTION FOR GWO
# ==============================================================

cv = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

fitness_cache = {}


def evaluate_fitness(mask):

    mask = np.asarray(mask, dtype=int)

    selected = np.flatnonzero(mask)

    if len(selected) == 0:
        return 1.0

    key = tuple(mask.tolist())

    if key in fitness_cache:
        return fitness_cache[key]

    X_subset = X_train.iloc[:, selected]

    model = create_knn()

    scores = cross_val_score(
        model,
        X_subset,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1
    )

    classification_error = 1.0 - scores.mean()

    feature_ratio = len(selected) / X_train.shape[1]

    fitness = (
        ALPHA * classification_error
        + BETA * feature_ratio
    )

    fitness_cache[key] = fitness

    return fitness


# ==============================================================
# 6. FAST FITNESS FUNCTION FOR RANDOM DIFFERENTIAL GROUPING
# ==============================================================

# Use a smaller validation split for grouping.
# This avoids repeatedly running 5-fold CV for every feature pair.

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    stratify=y_train,
    random_state=RANDOM_STATE
)

fast_fitness_cache = {}


def evaluate_fitness_fast(mask):

    mask = np.asarray(mask, dtype=int)

    selected = np.flatnonzero(mask)

    if len(selected) == 0:
        return 1.0

    key = tuple(mask.tolist())

    if key in fast_fitness_cache:
        return fast_fitness_cache[key]

    model = create_knn()

    model.fit(
        X_fit.iloc[:, selected],
        y_fit
    )

    predictions = model.predict(
        X_val.iloc[:, selected]
    )

    classification_error = 1.0 - accuracy_score(
        y_val,
        predictions
    )

    feature_ratio = len(selected) / X_train.shape[1]

    fitness = (
        ALPHA * classification_error
        + BETA * feature_ratio
    )

    fast_fitness_cache[key] = fitness

    return fitness


# ==============================================================
# 7. RANDOM DIFFERENTIAL GROUPING
# ==============================================================

def random_differential_grouping():

    n_features = X_train.shape[1]

    parent = list(range(n_features))

    def find(x):

        if parent[x] != x:
            parent[x] = find(parent[x])

        return parent[x]

    def union(a, b):

        root_a = find(a)
        root_b = find(b)

        if root_a != root_b:
            parent[root_b] = root_a

    print("\n" + "=" * 60)
    print("RANDOM DIFFERENTIAL GROUPING")
    print("=" * 60)

    total_pairs = n_features * (n_features - 1) // 2

    checked_pairs = 0

    for i in range(n_features):

        print(
            f"Checking feature {i + 1}/{n_features}...",
            flush=True
        )

        for j in range(i + 1, n_features):

            interaction_scores = []

            for _ in range(N_RNDG_SAMPLES):

                # Random context vector
                context = rng.integers(
                    0, 2,
                    size=n_features
                )

                context[i] = 0
                context[j] = 0

                # Ensure the context is not empty
                if context.sum() == 0:

                    available = [
                        k for k in range(n_features)
                        if k != i and k != j
                    ]

                    context[rng.choice(available)] = 1

                # Evaluate four combinations
                f00 = evaluate_fitness_fast(context)

                mask_i = context.copy()
                mask_i[i] = 1
                f10 = evaluate_fitness_fast(mask_i)

                mask_j = context.copy()
                mask_j[j] = 1
                f01 = evaluate_fitness_fast(mask_j)

                mask_both = context.copy()
                mask_both[i] = 1
                mask_both[j] = 1
                f11 = evaluate_fitness_fast(mask_both)

                # Differential interaction
                interaction = abs(
                    f11 - f10 - f01 + f00
                )

                interaction_scores.append(interaction)

            mean_interaction = np.mean(
                interaction_scores
            )

            if mean_interaction > INTERACTION_THRESHOLD:

                union(i, j)

            checked_pairs += 1

    # Construct groups
    groups = {}

    for i in range(n_features):

        root = find(i)

        groups.setdefault(root, []).append(i)

    result = list(groups.values())

    print("\nTotal feature pairs checked:", checked_pairs)

    print("\nIdentified feature groups:")

    for number, group in enumerate(result, 1):

        names = X.columns[group].tolist()

        print(f"Group {number}: {names}")

    print("\nTotal groups identified:", len(result))

    return result


# ==============================================================
# 8. BINARY GREY WOLF OPTIMIZATION
# ==============================================================

def sigmoid(x):

    x = np.clip(x, -20, 20)

    return 1.0 / (1.0 + np.exp(-x))


def binary_conversion(position):

    probabilities = sigmoid(position)

    mask = (
        rng.random(len(position)) < probabilities
    ).astype(int)

    # Prevent empty feature subsets
    if mask.sum() == 0:

        mask[np.argmax(probabilities)] = 1

    return mask


# ==============================================================
# 9. GROUP-AWARE POPULATION INITIALIZATION
# ==============================================================

def initialize_population(
    n_wolves,
    n_features,
    groups
):

    population = rng.uniform(
        -1,
        1,
        size=(n_wolves, n_features)
    )

    for wolf in range(n_wolves):

        for group in groups:

            if rng.random() < 0.5:

                selected = rng.choice(group)

                population[wolf, group] = -1.0

                population[wolf, selected] = 1.0

    return population


# ==============================================================
# 10. GREY WOLF OPTIMIZATION
# ==============================================================

def binary_gwo(groups):

    n_features = X_train.shape[1]

    population = initialize_population(
        N_WOLVES,
        n_features,
        groups
    )

    alpha_pos = np.zeros(n_features)
    beta_pos = np.zeros(n_features)
    delta_pos = np.zeros(n_features)

    alpha_score = float("inf")
    beta_score = float("inf")
    delta_score = float("inf")

    best_mask = None

    print("\n" + "=" * 60)
    print("GREY WOLF OPTIMIZATION")
    print("=" * 60)

    for iteration in range(MAX_ITERATIONS):

        # ------------------------------------------------------
        # Evaluate all wolves
        # ------------------------------------------------------

        for wolf_index in range(N_WOLVES):

            mask = binary_conversion(
                population[wolf_index]
            )

            score = evaluate_fitness(mask)

            # Alpha wolf
            if score < alpha_score:

                delta_score = beta_score
                delta_pos = beta_pos.copy()

                beta_score = alpha_score
                beta_pos = alpha_pos.copy()

                alpha_score = score
                alpha_pos = population[wolf_index].copy()

                best_mask = mask.copy()

            # Beta wolf
            elif score < beta_score:

                delta_score = beta_score
                delta_pos = beta_pos.copy()

                beta_score = score
                beta_pos = population[wolf_index].copy()

            # Delta wolf
            elif score < delta_score:

                delta_score = score
                delta_pos = population[wolf_index].copy()

        # ------------------------------------------------------
        # Update wolf positions
        # ------------------------------------------------------

        a = 2.0 - (
            2.0 * iteration / MAX_ITERATIONS
        )

        for wolf_index in range(N_WOLVES):

            X1 = np.zeros(n_features)
            X2 = np.zeros(n_features)
            X3 = np.zeros(n_features)

            leaders = [
                alpha_pos,
                beta_pos,
                delta_pos
            ]

            results = [X1, X2, X3]

            for leader, result in zip(leaders, results):

                r1 = rng.random(n_features)
                r2 = rng.random(n_features)

                A = 2.0 * a * r1 - a
                C = 2.0 * r2

                D = abs(
                    C * leader - population[wolf_index]
                )

                result[:] = leader - A * D

            population[wolf_index] = (
                X1 + X2 + X3
            ) / 3.0

        # ------------------------------------------------------
        # Preserve global best binary solution
        # ------------------------------------------------------

        print(
            f"Iteration {iteration + 1}/{MAX_ITERATIONS} | "
            f"Best fitness: {alpha_score:.5f} | "
            f"Selected features: {best_mask.sum()}",
            flush=True
        )

    return best_mask, alpha_score


# ==============================================================
# 11. RUN FEATURE GROUPING AND OPTIMIZATION
# ==============================================================

start_time = time.time()

groups = random_differential_grouping()

best_mask, best_fitness = binary_gwo(groups)

optimization_time = time.time() - start_time


# ==============================================================
# 12. DISPLAY SELECTED FEATURES
# ==============================================================

selected_indices = np.flatnonzero(best_mask)

selected_features = X.columns[selected_indices]

print("\n" + "=" * 60)
print("SELECTED FEATURES")
print("=" * 60)

for feature in selected_features:

    print(feature)

print("\nSelected feature count:", len(selected_features))

print("Best fitness:", best_fitness)


# ==============================================================
# 13. BASELINE KNN
# ==============================================================

baseline_model = create_knn()

baseline_model.fit(
    X_train,
    y_train
)

baseline_predictions = baseline_model.predict(
    X_test
)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)


# ==============================================================
# 14. OPTIMIZED KNN
# ==============================================================

X_train_selected = X_train.iloc[:, selected_indices]

X_test_selected = X_test.iloc[:, selected_indices]

optimized_model = create_knn()

optimized_model.fit(
    X_train_selected,
    y_train
)

optimized_predictions = optimized_model.predict(
    X_test_selected
)

optimized_accuracy = accuracy_score(
    y_test,
    optimized_predictions
)

precision = precision_score(
    y_test,
    optimized_predictions,
    zero_division=0
)

recall = recall_score(
    y_test,
    optimized_predictions,
    zero_division=0
)

f1 = f1_score(
    y_test,
    optimized_predictions,
    zero_division=0
)


# ==============================================================
# 15. FEATURE REDUCTION
# ==============================================================

original_features = X.shape[1]

selected_count = len(selected_indices)

feature_reduction = (
    (original_features - selected_count)
    / original_features
) * 100


# ==============================================================
# 16. FINAL RESULTS
# ==============================================================

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print("Dataset: Breast Cancer")

print(
    "Original number of features:",
    original_features
)

print(
    "Selected number of features:",
    selected_count
)

print(
    f"Baseline KNN accuracy: "
    f"{baseline_accuracy * 100:.2f}%"
)

print(
    f"DrnDG-GWO + KNN accuracy: "
    f"{optimized_accuracy * 100:.2f}%"
)

print(
    f"Feature reduction: "
    f"{feature_reduction:.2f}%"
)

print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1-score: {f1 * 100:.2f}%")

print(
    f"Optimization runtime: "
    f"{optimization_time:.2f} seconds"
)


# ==============================================================
# 17. CLASSIFICATION REPORT
# ==============================================================

print("\nCLASSIFICATION REPORT")

print(
    classification_report(
        y_test,
        optimized_predictions,
        target_names=dataset.target_names,
        zero_division=0
    )
)

print("CONFUSION MATRIX")

print(
    confusion_matrix(
        y_test,
        optimized_predictions
    )
)


# ==============================================================
# 18. FEATURE SELECTION SUMMARY
# ==============================================================

summary = pd.DataFrame({
    "Feature": X.columns,
    "Selected": best_mask.astype(bool)
})

print("\nFEATURE SELECTION SUMMARY")

print(
    summary.to_string(index=False)
)


# ==============================================================
# 19. RESULTS TABLE
# ==============================================================

results = pd.DataFrame({
    "Metric": [
        "Total Features",
        "Selected Features",
        "Feature Reduction (%)",
        "Baseline Accuracy (%)",
        "Optimized Accuracy (%)",
        "Precision (%)",
        "Recall (%)",
        "F1 Score (%)",
        "Optimization Time (seconds)"
    ],

    "Value": [
        original_features,
        selected_count,
        round(feature_reduction, 2),
        round(baseline_accuracy * 100, 2),
        round(optimized_accuracy * 100, 2),
        round(precision * 100, 2),
        round(recall * 100, 2),
        round(f1 * 100, 2),
        round(optimization_time, 2)
    ]
})

print("\nFINAL EXPERIMENT TABLE")

print(results.to_string(index=False))


DATASET INFORMATION
Dataset: Breast Cancer
Total samples: 569
Total features: 30

Training samples: 455
Testing samples: 114

RANDOM DIFFERENTIAL GROUPING
Checking feature 1/30...
Checking feature 2/30...
Checking feature 3/30...
Checking feature 4/30...
Checking feature 5/30...
Checking feature 6/30...
Checking feature 7/30...
Checking feature 8/30...
Checking feature 9/30...
Checking feature 10/30...
Checking feature 11/30...
Checking feature 12/30...
Checking feature 13/30...
Checking feature 14/30...
Checking feature 15/30...
Checking feature 16/30...
Checking feature 17/30...
Checking feature 18/30...
Checking feature 19/30...
Checking feature 20/30...
Checking feature 21/30...
Checking feature 22/30...
Checking feature 23/30...
Checking feature 24/30...
Checking feature 25/30...
Checking feature 26/30...
Checking feature 27/30...
Checking feature 28/30...
Checking feature 29/30...
Checking feature 30/30...

Total feature pairs checked: 435

Identified feature groups:
Group 1: ['m